# DFT Density Grid

This notebook inspects the first DFT building blocks: a real-space grid, a toy local pseudopotential, normalized orbitals, electron density `ρ(r)`, Hartree potential, and LDA exchange. The values are in atomic units and the example is deliberately small.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.dft import (
    LocalGaussianPseudopotential,
    RealSpaceGrid,
    density_from_orbitals,
    hartree_potential,
    lda_exchange_energy_potential,
    normalize_orbitals,
)

The grid is periodic and orthorhombic. Each grid point represents a small volume element `dr³`, so integrating a density means summing the grid values and multiplying by `grid.dv`.

In [ ]:
grid = RealSpaceGrid((32, 32, 32), [8.0, 8.0, 8.0])
grid.volume, grid.dv, np.array(grid.spacing)

A Gaussian pseudopotential is only a toy external potential. It is useful here because we can see a smooth attractive well without needing a real pseudopotential file format yet.

In [ ]:
local = LocalGaussianPseudopotential(
    centers=[[4.0, 4.0, 4.0]],
    amplitudes=-3.0,
    widths=0.8,
)
v_local = local.field(grid)

coords = np.array(grid.coordinates())
trial = np.exp(-np.sum((coords - 4.0) ** 2, axis=-1) / 1.2).astype(np.float32)
orbital = normalize_orbitals(trial, grid)
rho = density_from_orbitals(orbital, grid)

electron_count = float(np.sum(np.array(rho)) * grid.dv)
electron_count

For a closed-shell single orbital, the default occupation is `2`, so the density integrates to two electrons. The slice below shows the local potential and the density generated by the normalized orbital.

In [ ]:
mid = grid.shape[2] // 2
fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)

im0 = axes[0].imshow(np.array(v_local)[:, :, mid], origin="lower", cmap="viridis")
axes[0].set_title("local potential")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(np.array(rho)[:, :, mid], origin="lower", cmap="magma")
axes[1].set_title("electron density ρ(r)")
fig.colorbar(im1, ax=axes[1], fraction=0.046);

The Hartree potential is solved in reciprocal space. The `G = 0` component is removed, which is the usual neutralizing-background convention for this small periodic toy model. LDA exchange is the Dirac exchange-only expression.

In [ ]:
v_h = hartree_potential(rho, grid)
e_x, v_x = lda_exchange_energy_potential(rho, grid)

{
    "hartree_min": float(np.min(np.array(v_h))),
    "hartree_max": float(np.max(np.array(v_h))),
    "exchange_energy": float(e_x),
    "exchange_potential_min": float(np.min(np.array(v_x))),
}